# RescueVision - FloodNet train-cell repro (CPU)

In [ ]:
import subprocess, sys, os, json, time, traceback
from pathlib import Path

WORK = Path("/kaggle/working")
REPO = Path("/tmp/RescueVision")
if not REPO.exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/AliOmarAbdelhady/RescueVision.git", str(REPO)], check=True)
sys.path.insert(0, str(REPO / "src"))

def mark(stage, ok=True, extra=""):
    """Progress ledger in /kaggle/working: survives kills where the log does not."""
    rec = {"t": time.strftime("%H:%M:%S"), "stage": stage, "ok": bool(ok), "extra": str(extra)[:1500]}
    with open(WORK / "_progress.txt", "a") as f:
        f.write(json.dumps(rec) + "\n")
    (WORK / "_status.json").write_text(json.dumps(rec, indent=1))
    print("[mark]", rec, flush=True)

mark("setup:start")
# pin albumentations: newer versions (preinstalled on the GPU image) have a
# stricter shape check that rejects our verified-consistent (HxWx3, HxW) pairs;
# --upgrade forces the downgrade past pip's "already satisfied"; all transforms
# ALSO use is_check_shapes=False as a version-proof second layer
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
                "albumentations==2.0.8"], check=False)
import albumentations as _A
mark("setup:albumentations", extra=_A.__version__)
print("albumentations", _A.__version__)

import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# num_workers=0: DataLoader worker processes crash on the Kaggle GPU image
# (both floodnet v7 and v8 died at the first batch in a worker). Loading on the
# main process costs only a few percent of step time on /tmp-resident data.
NW = 0
try:
    torch.multiprocessing.set_sharing_strategy("file_system")
except Exception:
    pass
print("torch", torch.__version__, "| device:", DEVICE)
print(subprocess.run(["df", "-h", "/tmp", "/kaggle/working"], capture_output=True, text=True).stdout)
mark("setup:done", extra=str(DEVICE))

# per-epoch heartbeat: every trainer that logs an epoch also lands in the marks,
# so a silent kill still tells us how far training got and with what metrics
import rescuevision.training.logging_utils as _lu
_orig_log = _lu.CSVLogger.log
def _marked_log(self, metrics, epoch):
    try:
        mark(f"epoch{epoch}", extra=str({k: round(float(v), 4) for k, v in metrics.items()
                                         if isinstance(v, (int, float))})[:400])
    except Exception:
        pass
    return _orig_log(self, metrics, epoch)
_lu.CSVLogger.log = _marked_log
print("epoch heartbeat patch applied")

In [ ]:
import numpy as np
import torch
import albumentations as A
from albumentations.pytorch import ToTensorV2

from rescuevision.constants import IMAGENET_MEAN, IMAGENET_STD
from rescuevision.data import datasets as _D
from PIL import Image as _PIL
from pathlib import Path as _PPath

def _load_arr(path):
    """Read .png or .npy array transparently."""
    path = _PPath(path)
    if path.suffix == ".npy":
        return np.load(path)
    return np.array(_PIL.open(path))

def _find_tile(directory, tile_id):
    for ext in (".png", ".npy"):
        p = _PPath(directory) / f"{tile_id}{ext}"
        if p.exists():
            return p
    raise FileNotFoundError(f"{tile_id} in {directory}")

def joint_seg_transforms(cfg, phase="train", image_size=512):
    """Joint pipeline for a stacked pre+post pair (H, W, 6): one geometric draw for both."""
    mean, std = IMAGENET_MEAN * 2, IMAGENET_STD * 2
    tf = [A.Resize(image_size, image_size)]
    if phase == "train":
        if cfg.get("horizontal_flip", 0) > 0:
            tf.append(A.HorizontalFlip(p=cfg["horizontal_flip"]))
        if cfg.get("vertical_flip", 0) > 0:
            tf.append(A.VerticalFlip(p=cfg["vertical_flip"]))
        if cfg.get("rotate90", 0) > 0:
            tf.append(A.RandomRotate90(p=cfg["rotate90"]))
        if cfg.get("brightness_contrast", 0) > 0:
            tf.append(A.RandomBrightnessContrast(p=cfg["brightness_contrast"]))
    if cfg.get("normalize", True):
        tf.append(A.Normalize(mean=mean, std=std))
    tf.append(ToTensorV2())
    return A.Compose(tf, is_check_shapes=False)

def joint_clf_pair_transforms(cfg, phase="train", image_size=224):
    """Joint pipeline for stacked pre+post crops (H, W, 6)."""
    mean, std = IMAGENET_MEAN * 2, IMAGENET_STD * 2
    tf = [A.Resize(image_size, image_size)]
    if phase == "train":
        if cfg.get("horizontal_flip", 0) > 0:
            tf.append(A.HorizontalFlip(p=cfg["horizontal_flip"]))
        if cfg.get("vertical_flip", 0) > 0:
            tf.append(A.VerticalFlip(p=cfg["vertical_flip"]))
        if cfg.get("rotate", 0) > 0:
            tf.append(A.Rotate(limit=cfg["rotate"], p=0.5))
        if cfg.get("brightness_contrast", 0) > 0:
            tf.append(A.RandomBrightnessContrast(p=cfg["brightness_contrast"]))
    if cfg.get("normalize", True):
        tf.append(A.Normalize(mean=mean, std=std))
    tf.append(ToTensorV2())
    return A.Compose(tf, is_check_shapes=False)

def _nocheck(tf):
    # disable albumentations' shape pre-check: newer builds (preinstalled on the
    # Kaggle GPU image) reject our verified HxW-consistent image/mask pairs
    try:
        return A.Compose(tf.transforms, is_check_shapes=False)
    except Exception:
        return tf

def _joint_dmg_seg_getitem(self, idx):
    tile_id = self.tile_ids[idx]
    pre_image = _load_arr(_find_tile(self.pre_dir, tile_id))
    post_image = _load_arr(_find_tile(self.post_dir, tile_id))
    mask = _load_arr(_find_tile(self.masks_dir, tile_id))
    pair = np.concatenate([pre_image, post_image], axis=2)
    if self.transform:
        augmented = self.transform(image=pair, mask=mask)
        image, mask = augmented["image"], augmented["mask"]
    else:
        image = torch.from_numpy(pair.transpose(2, 0, 1))
    mask = mask.long() if torch.is_tensor(mask) else torch.from_numpy(mask).long()
    return {"image": image, "mask": mask, "tile_id": tile_id}

def _joint_change_getitem(self, idx):
    tile_id = self.tile_ids[idx]
    pre_image = _load_arr(_find_tile(self.pre_dir, tile_id))
    post_image = _load_arr(_find_tile(self.post_dir, tile_id))
    mask = _load_arr(_find_tile(self.masks_dir, tile_id))
    pair = np.concatenate([pre_image, post_image], axis=2)
    if self.transform:
        augmented = self.transform(image=pair, mask=mask)
        pair_t, mask = augmented["image"], augmented["mask"]
    else:
        pair_t = torch.from_numpy(pair.transpose(2, 0, 1))
    mask = mask.long() if torch.is_tensor(mask) else torch.from_numpy(mask).long()
    return {"pre_image": pair_t[:3], "post_image": pair_t[3:], "mask": mask, "tile_id": tile_id}

_D.XBDDamageSegmentationDataset.__getitem__ = _joint_dmg_seg_getitem
_D.XBDChangeDetectionDataset.__getitem__ = _joint_change_getitem

def _tile_dataset_init(self, processed_dir, split_csv, split="train", mask_type="building_binary",
                       transform=None, image_size=512, debug_limit=None):
    import pandas as pd
    self.processed_dir = _PPath(processed_dir)
    self.images_dir = self.processed_dir / "images" / "pre"
    self.pre_dir = self.processed_dir / "images" / "pre"
    self.post_dir = self.processed_dir / "images" / "post"
    if type(self).__name__ == "XBDDamageSegmentationDataset":
        self.masks_dir = self.processed_dir / "masks" / "damage_multiclass"
    elif type(self).__name__ == "XBDChangeDetectionDataset":
        self.masks_dir = self.processed_dir / "masks" / "change_binary"
    else:
        self.masks_dir = self.processed_dir / "masks" / mask_type
    self.transform = transform
    self.image_size = image_size
    splits_df = pd.read_csv(split_csv)
    tile_ids = splits_df[splits_df["split"] == split]["tile_id"].tolist()
    if debug_limit:
        tile_ids = tile_ids[:debug_limit]
    self.tile_ids = [t for t in tile_ids
                     if _find_tile(self.pre_dir, t).exists()
                     and _find_tile(self.post_dir, t).exists()
                     and _find_tile(self.masks_dir, t).exists()]
    print(f"{type(self).__name__}: split={split}, {len(self.tile_ids)} tiles")

def _xbd_seg_getitem(self, idx):
    tile_id = self.tile_ids[idx]
    image = _load_arr(_find_tile(self.pre_dir, tile_id))
    mask = _load_arr(_find_tile(self.masks_dir, tile_id))
    if self.transform:
        aug = self.transform(image=image, mask=mask)
        image, mask = aug["image"], aug["mask"]
    mask = mask.long() if torch.is_tensor(mask) else torch.from_numpy(mask).long()
    return {"image": image, "mask": mask, "tile_id": tile_id}

_D.XBDSegmentationDataset.__init__ = _tile_dataset_init
_D.XBDDamageSegmentationDataset.__init__ = _tile_dataset_init
_D.XBDChangeDetectionDataset.__init__ = _tile_dataset_init
_D.XBDSegmentationDataset.__getitem__ = _xbd_seg_getitem
mark("patch:done")
print("png-aware dataset init patch applied")
print("joint pre/post transform patch applied")

In [ ]:
import numpy as np, pandas as pd, random
from PIL import Image
from pathlib import Path

ROOT = None
for _c in Path("/kaggle/input").rglob("class_mapping.csv"):
    ROOT = _c.parent
    break
if ROOT is None and Path("/tmp/floodnet_src").exists():
    ROOT = Path("/tmp/floodnet_src")          # synthetic smoke path
assert ROOT is not None, "FloodNet dataset not found under /kaggle/input"
print("FloodNet root:", ROOT)
mark("floodnet:root", extra=str(ROOT))
TILES = Path("/tmp/floodnet_tiles")
(TILES / "train").mkdir(parents=True, exist_ok=True)
(TILES / "val").mkdir(parents=True, exist_ok=True)

from rescuevision.constants import FLOODNET_CLASSES, NUM_FLOODNET_CLASSES

import re
def norm_name(s):
    return re.sub(r"[^a-z]", "", str(s).lower())

NAME_TO_ID = {norm_name(v): k for k, v in FLOODNET_CLASSES.items()}

OFFICIAL_PALETTE = {  # official FloodNet colors, fallback if the CSV can't be parsed
    (0, 0, 0): 0, (72, 0, 255): 1, (255, 0, 0): 2, (0, 255, 255): 3, (255, 255, 0): 4,
    (0, 255, 0): 5, (0, 0, 255): 6, (255, 0, 255): 7, (0, 128, 128): 8, (128, 128, 128): 9,
}

color_to_id = {}
try:
    cm = pd.read_csv(ROOT / "class_mapping.csv")
    print("class_mapping.csv:\n", cm.to_string())
    cols = {c.lower(): c for c in cm.columns}
    name_col = next((cols[c] for c in cols if "name" in c or "class" in c), None)
    has_rgb = all(any(k in cols for k in (key,)) for key in ("r", "g", "b"))
    hex_col = next((cols[c] for c in cols if "hex" in c or "color" in c), None)
    for _, row in cm.iterrows():
        cid = NAME_TO_ID.get(norm_name(row[name_col])) if name_col else None
        if cid is None:
            continue
        if has_rgb:
            color_to_id[(int(row[cols["r"]]), int(row[cols["g"]]), int(row[cols["b"]]))] = cid
        elif hex_col and str(row[hex_col]).startswith("#"):
            h = str(row[hex_col]).lstrip("#")
            color_to_id[tuple(int(h[i:i+2], 16) for i in (0, 2, 4))] = cid
except Exception as e:
    print("could not parse class_mapping.csv:", e)
if len(color_to_id) < NUM_FLOODNET_CLASSES and (ROOT / "class_mapping.csv").exists() is False:
    print("no class_mapping.csv - masks assumed grayscale ids")
elif len(color_to_id) < NUM_FLOODNET_CLASSES:
    print("falling back to official FloodNet palette")
    color_to_id = OFFICIAL_PALETTE

def mask_to_ids(mask_rgb):
    # (H, W, 3) uint8 -> (H, W) class ids, nearest palette color
    m = mask_rgb.astype(np.int16)
    ids = np.zeros(m.shape[:2], dtype=np.uint8)
    best = np.full(m.shape[:2], 3 * 255, dtype=np.int16)
    for color, cid in color_to_id.items():
        d = (np.abs(m[:, :, 0] - color[0]) + np.abs(m[:, :, 1] - color[1])
             + np.abs(m[:, :, 2] - color[2])).astype(np.int16)
        better = d < best
        ids[better] = cid
        np.minimum(best, d, out=best)
    return ids

sample_mask = next((ROOT / "Train/Labeled/Flooded/mask").glob("*.png"))
sm = Image.open(sample_mask)
print("sample mask mode:", sm.mode, sm.size)
grayscale_mode = sm.mode in ("L", "P")
print("grayscale_mode:", grayscale_mode)
mark("floodnet:format", extra=f"grayscale={grayscale_mode}")

In [ ]:
import random, os
from tqdm.auto import tqdm

TILE = 512
SEED = 42
VAL_FRAC = 0.15
MAX_TRAIN_TILES = 1000000000    # quota-friendly caps (flood tiles kept with priority)
MAX_VAL_TILES = 1000000000
MAX_TMP_GB = 40       # /tmp safety guard

img_dirs = [ROOT / "Train/Labeled/Flooded", ROOT / "Train/Labeled/Non-Flooded"]
samples = []
for d in img_dirs:
    for img_path in sorted((d / "image").glob("*.jpg")):
        mask_path = d / "mask" / (img_path.stem + "_lab.png")   # official naming: {id}_lab.png
        if not mask_path.exists():
            mask_path = d / "mask" / (img_path.stem + ".png")
        if mask_path.exists():
            samples.append((img_path, mask_path))
print(f"{len(samples)} labeled images")
mark("floodnet:samples", extra=str(len(samples)))

rng = random.Random(SEED)
idx = rng.sample(range(len(samples)), max(1, int(len(samples) * VAL_FRAC)))
val_set = set(idx)

def tree_bytes(p):
    return sum(f.stat().st_size for f in Path(p).rglob("*") if f.is_file())

records = []   # (img_npy, ids_npy, split, is_flood, bincount_row)
n_train = n_val = 0
for i, (img_path, mask_path) in enumerate(tqdm(samples, desc="tiling")):
    split = "val" if i in val_set else "train"
    img = np.array(Image.open(img_path).convert("RGB"))
    msk_img = Image.open(mask_path)
    if grayscale_mode:
        ids = np.array(msk_img.convert("L"), dtype=np.uint8)
    else:
        ids = mask_to_ids(np.array(msk_img.convert("RGB")))
    ids = np.clip(ids, 0, NUM_FLOODNET_CLASSES - 1)  # guard stray values
    H, W = img.shape[:2]
    for y in range(0, H - TILE + 1, TILE):
        for x in range(0, W - TILE + 1, TILE):
            tile_ids = ids[y:y+TILE, x:x+TILE]
            if len(np.unique(tile_ids)) < 2:
                continue  # skip fully-uniform tiles (usually all background)
            stem = f"{img_path.stem}_{y//TILE}_{x//TILE}"
            img_npy = TILES / split / f"{stem}_img.npy"
            ids_npy = TILES / split / f"{stem}_ids.npy"
            np.save(img_npy, img[y:y+TILE, x:x+TILE])
            np.save(ids_npy, tile_ids)
            cnt = np.bincount(tile_ids.flatten(), minlength=NUM_FLOODNET_CLASSES)
            records.append((img_npy, ids_npy, split, bool(cnt[1] + cnt[3] + cnt[5] > 0), cnt))
            if split == "train": n_train += 1
            else: n_val += 1
    if (i + 1) % 200 == 0 and tree_bytes(TILES) > MAX_TMP_GB * 1024**3:
        print(f"/tmp size guard hit at image {i+1}, stopping tiling early")
        mark("floodnet:tiling-guard", extra=str(i))
        break

print(f"tiles before cap: train={n_train} val={n_val}")

# cap train tiles: keep ALL flood tiles, subsample the rest deterministically
rng2 = np.random.default_rng(SEED)
train_recs = [r for r in records if r[2] == "train"]
if len(train_recs) > MAX_TRAIN_TILES:
    flood = [r for r in train_recs if r[3]]
    rest = [r for r in train_recs if not r[3]]
    budget = max(0, MAX_TRAIN_TILES - len(flood))
    keep = set(map(id, flood)) | set(map(id, rng2.choice(rest, size=min(budget, len(rest)), replace=False) if len(rest) else []))
    for r in train_recs:
        if id(r) not in keep:
            r[0].unlink(missing_ok=True); r[1].unlink(missing_ok=True)
    train_recs = [r for r in train_recs if id(r) in keep]
val_recs = [r for r in records if r[2] == "val"]
if len(val_recs) > MAX_VAL_TILES:
    keep = set(map(id, rng2.choice(val_recs, size=MAX_VAL_TILES, replace=False)))
    for r in val_recs:
        if id(r) not in keep:
            r[0].unlink(missing_ok=True); r[1].unlink(missing_ok=True)
    val_recs = [r for r in val_recs if id(r) in keep]

counts = np.sum([r[4] for r in train_recs], axis=0).astype(np.int64) if train_recs else np.zeros(NUM_FLOODNET_CLASSES, dtype=np.int64)
print(f"tiles after cap: train={len(train_recs)} val={len(val_recs)} | tree {tree_bytes(TILES)/1e9:.1f} GB")
print("class pixel counts:", counts.tolist())
assert counts.sum() > 0
mark("floodnet:tiles", extra=f"train={len(train_recs)} val={len(val_recs)}")

In [ ]:
import logging
import pandas as pd
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from rescuevision.models.losses import WeightedCEDiceLoss
from rescuevision.training.train_segmentation import train_segmentation, evaluate_segmentation
from rescuevision.utils.seed import set_seed
mark("train:imports")

TILES = Path("/tmp/floodnet_tiles")
SEED = 42
AUG = {"horizontal_flip": 0.5, "vertical_flip": 0.5, "rotate90": 0.5, "brightness_contrast": 0.3, "normalize": True}

class FloodNetTileDataset(Dataset):
    def __init__(self, split_dir, transform=None):
        self.files = sorted(split_dir.glob("*_img.npy"))
        self.transform = transform
    def __len__(self):
        return len(self.files)
    def __getitem__(self, i):
        base = str(self.files[i])[: -len("_img.npy")]
        image = np.load(base + "_img.npy")
        mask = np.load(base + "_ids.npy")
        if self.transform:
            aug = self.transform(image=image, mask=mask)
            image, mask = aug["image"], aug["mask"]
        mask = mask.long() if torch.is_tensor(mask) else torch.from_numpy(mask).long()
        return {"image": image, "mask": mask, "tile_id": Path(base).name}

from rescuevision.data.transforms import get_segmentation_transforms
mark("train:pre-data")

set_seed(SEED)
train_ds = FloodNetTileDataset(TILES / "train", _nocheck(get_segmentation_transforms(AUG, "train", 512)))
val_ds = FloodNetTileDataset(TILES / "val", _nocheck(get_segmentation_transforms(AUG, "val", 512)))
print(f"train={len(train_ds)} val={len(val_ds)}")
assert len(train_ds) > 0 and len(val_ds) > 0
train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=0, pin_memory=False)
val_loader = DataLoader(val_ds, batch_size=4, shuffle=False, num_workers=0, pin_memory=False)
mark("train:data", extra=f"train={len(train_ds)} val={len(val_ds)}")

counts = np.zeros(10, dtype=np.int64)
for f in list((TILES / "train").glob("*_ids.npy"))[:300]:
    counts += np.bincount(np.load(f).flatten(), minlength=10)
freq = counts / counts.sum()
weights = 1.0 / np.sqrt(freq)
weights = weights / weights.sum() * 10
class_weights = torch.tensor(weights, dtype=torch.float32, device=DEVICE)
mark("train:weights", extra=str([round(w, 2) for w in weights.tolist()]))

from transformers import SegformerForSemanticSegmentation
segformer = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/segformer-b2-finetuned-ade-512-512", num_labels=10, ignore_mismatched_sizes=True)
mark("train:model-downloaded")

class SegFormerWrapper(torch.nn.Module):
    def __init__(self, m):
        super().__init__()
        self.model = m
    def forward(self, x):
        logits = self.model(x).logits
        return F.interpolate(logits, size=x.shape[2:], mode="bilinear", align_corners=False)

model = SegFormerWrapper(segformer).to(DEVICE)
loss_fn = WeightedCEDiceLoss(class_weights=class_weights, ce_weight=0.5, dice_weight=0.5, ignore_index=255)
mark("train:model-on-device", extra=str(DEVICE))

# probe: 2 batches fwd+bwd
import traceback
try:
    _it = iter(train_loader)
    for _ in range(2):
        _b = next(_it)
        _lg = model(_b["image"].to(DEVICE))
        _loss = loss_fn(_lg, _b["mask"].to(DEVICE))
        _loss.backward()
        model.zero_grad(set_to_none=True)
        print("probe step loss:", float(_loss))
    mark("train:probe-ok")
except Exception:
    mark("train:probe-fail", extra=traceback.format_exc()[-1200:])
    raise

# 5 real optimizer steps (AMP off on CPU)
opt = torch.optim.AdamW(model.parameters(), lr=6e-5)
scaler = torch.amp.GradScaler(DEVICE.type, enabled=(DEVICE.type == "cuda"))
try:
    model.train()
    for i, _b in enumerate(train_loader):
        if i >= 5:
            break
        with torch.amp.autocast(DEVICE.type, enabled=(DEVICE.type == "cuda")):
            _lg = model(_b["image"].to(DEVICE))
            _loss = loss_fn(_lg, _b["mask"].to(DEVICE)) / 2
        scaler.scale(_loss).backward()
        scaler.step(opt)
        scaler.update()
        opt.zero_grad()
        print(f"step {i}: loss {float(_loss):.4f}")
    mark("train:steps-ok", extra="5 optimizer steps")
except Exception:
    mark("train:steps-fail", extra=traceback.format_exc()[-1200:])
    raise

# val batch through evaluate_segmentation (repo path)
val_loader_small = DataLoader(val_ds, batch_size=4, shuffle=False, num_workers=0)
m = evaluate_segmentation(model, val_loader_small, DEVICE, is_binary=False, num_classes=10, ignore_index=255)
print("val metrics (1 batch-ish):", {k: round(v, 4) for k, v in m.items() if not isinstance(v, list)})
mark("train:eval-ok")
print("EXTENDED REPRO PASSED: full train-cell sequence works on", DEVICE)